# 重複・言い換えの文ペアを見つける

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/10_sentence_embeddings/paraphrase_mining.ipynb)

NLP クックブック 10-03 ／ 文埋め込みとクラスタリング

コーパスの中から、ほぼ同じ内容を述べている文のペアを洗い出したい（FAQ の整理、データの重複除去）。

In [1]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("intfloat/multilingual-e5-small")
sentences = [
    "返品はいつまで可能ですか？", "商品を返したいのですが期限はありますか。", "送料はいくらかかりますか？",
    "配送料を教えてください。", "支払い方法には何がありますか。", "クレジットカードは使えますか？",
    "返品の締め切りを知りたい。", "営業時間は何時から何時までですか？", "お店は何時に開きますか。",
]
pairs = util.paraphrase_mining(model, ["query: " + s for s in sentences], top_k=3)
for score, i, j in pairs[:8]:
    print(f"{score:.3f}  {sentences[i]}  ⇔  {sentences[j]}")

0.933  返品はいつまで可能ですか？  ⇔  商品を返したいのですが期限はありますか。
0.913  営業時間は何時から何時までですか？  ⇔  お店は何時に開きますか。
0.910  商品を返したいのですが期限はありますか。  ⇔  返品の締め切りを知りたい。
0.898  送料はいくらかかりますか？  ⇔  配送料を教えてください。
0.895  返品はいつまで可能ですか？  ⇔  返品の締め切りを知りたい。
0.876  配送料を教えてください。  ⇔  返品の締め切りを知りたい。
0.876  支払い方法には何がありますか。  ⇔  クレジットカードは使えますか？
0.870  商品を返したいのですが期限はありますか。  ⇔  配送料を教えてください。


In [2]:
# しきい値以上のペアをつないで、重複グループにまとめる
clusters = util.community_detection(model.encode(["query: " + s for s in sentences], convert_to_tensor=True),
                                    threshold=0.89, min_community_size=2)
for c in clusters:
    print([sentences[i] for i in c])

['返品はいつまで可能ですか？', '商品を返したいのですが期限はありますか。', '返品の締め切りを知りたい。']
['送料はいくらかかりますか？', '配送料を教えてください。']
['営業時間は何時から何時までですか？', 'お店は何時に開きますか。']


- `paraphrase_mining` は全ペアの総当たりを避けてチャンク単位で計算するので、数万文でも実行できる。